# Step 2: Data Partitioning
**TSU NSF AI Workshop 2026 — Federated Learning Lab**

---

## What are we doing in this notebook?

In federated learning, the data is **distributed** — it lives on separate machines (hospitals, clinics, phones) and never gets collected in one place.

In this notebook, we **simulate** two community health centers by splitting the DermaMNIST training data between them.

### What is Non-IID data?

| Term | Meaning |
|------|---------|
| **IID** | Every client has the same balanced mix of all classes |
| **Non-IID** | Each client has a *different* mix — some classes are rare at one site, common at another |

**Non-IID is realistic!**
A rural clinic might see mostly common moles (melanocytic nevi), while a specialist dermatology center sees more rare cancers.

Our split:
- **Client 1 (Health Center A):** Mostly rare lesion types (classes 0–3)
- **Client 2 (Health Center B):** Mostly common lesion types (classes 4–6)

This makes federated learning **harder** — and more realistic.

## 🔧 Mount Google Drive

We need access to:
- **Input:** `FederatedLearning/data/dermamnist.npz`
- **Output:** `FederatedLearning/data/client1.npz`, `client2.npz`, `test_data.npz`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np

DRIVE_BASE = "/content/drive/MyDrive/FederatedLearning"
DATA_PATH  = os.path.join(DRIVE_BASE, "data", "dermamnist.npz")
DATA_DIR   = os.path.join(DRIVE_BASE, "data")

print("Google Drive mounted!")
print(f"Looking for data at: {DATA_PATH}")
print("Found!" if os.path.exists(DATA_PATH) else "ERROR: File not found!")

## Step 1: Load the Training Data

We only need the training data here — the test set stays untouched for final evaluation.

In [ ]:
data = np.load(DATA_PATH)
train_images = data["train_images"]           # (7007, 28, 28, 3)
train_labels = data["train_labels"].flatten() # (7007,)

CLASS_NAMES = [
    "Actinic keratoses",    # Class 0  (rare)
    "Basal cell carcinoma", # Class 1  (rare)
    "Benign keratosis",     # Class 2
    "Dermatofibroma",       # Class 3  (rare)
    "Melanoma",             # Class 4
    "Melanocytic nevi",     # Class 5  (very common — 67% of data!)
    "Vascular lesions",     # Class 6  (rare)
]

print(f"Total training samples: {len(train_labels)}")
print("\nClass distribution (full dataset):")
for c, name in enumerate(CLASS_NAMES):
    n = np.sum(train_labels == c)
    pct = n / len(train_labels)
    bar = "█" * int(pct * 50)
    print(f"  Class {c} ({name:<25}): {n:4d} ({pct:.1%})  {bar}")

## Step 2: Split Data into 2 Clients (Non-IID)

We control how skewed the split is using `CLIENT1_FRACTION`:
- For **rare** lesion classes (0–3): Client 1 gets 80% of samples
- For **common** lesion classes (4–6): Client 1 gets only 20% of samples

Client 2 always gets whatever Client 1 doesn't.

This means the two clients have **very different patient populations** — exactly like real hospitals.

In [ ]:
# Fraction of each class that goes to Client 1
# (Client 2 gets the rest)
CLIENT1_FRACTION = {
    0: 0.8,  # Actinic keratoses    → Client 1 gets 80%
    1: 0.8,  # Basal cell carcinoma → Client 1 gets 80%
    2: 0.8,  # Benign keratosis     → Client 1 gets 80%
    3: 0.8,  # Dermatofibroma       → Client 1 gets 80%
    4: 0.2,  # Melanoma             → Client 1 gets 20%
    5: 0.2,  # Melanocytic nevi     → Client 1 gets 20%
    6: 0.2,  # Vascular lesions     → Client 1 gets 20%
}

np.random.seed(42)  # Fixed seed → same split every time you run this
client1_idx = []
client2_idx = []

for class_id in range(len(CLASS_NAMES)):
    idx = np.where(train_labels == class_id)[0]
    np.random.shuffle(idx)
    split = int(len(idx) * CLIENT1_FRACTION[class_id])
    client1_idx.extend(idx[:split])
    client2_idx.extend(idx[split:])

client1_idx = np.array(client1_idx)
client2_idx = np.array(client2_idx)

print(f"Client 1 total samples: {len(client1_idx)}")
print(f"Client 2 total samples: {len(client2_idx)}")

## Step 3: Visualize the Class Distribution

Let's see exactly how different the two clients' data looks.

In [ ]:
import matplotlib.pyplot as plt

c1_labels = train_labels[client1_idx]
c2_labels = train_labels[client2_idx]

# Text breakdown
print(f"{'Class':<28} {'Client 1':>10} {'Client 2':>10}  | Distribution")
print("-" * 70)
for c, name in enumerate(CLASS_NAMES):
    n1 = int(np.sum(c1_labels == c))
    n2 = int(np.sum(c2_labels == c))
    total = n1 + n2
    pct1  = n1 / total if total > 0 else 0
    bar   = "█" * int(pct1 * 20) + "░" * (20 - int(pct1 * 20))
    print(f"  {name:<26} {n1:>10} {n2:>10}  | C1|{bar}|C2")

# Bar chart
x     = np.arange(len(CLASS_NAMES))
n1_counts = [int(np.sum(c1_labels == c)) for c in range(len(CLASS_NAMES))]
n2_counts = [int(np.sum(c2_labels == c)) for c in range(len(CLASS_NAMES))]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.2, n1_counts, width=0.4, label="Client 1 (Health Center A)", color="steelblue")
ax.bar(x + 0.2, n2_counts, width=0.4, label="Client 2 (Health Center B)", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels([f"Class {c}\n{CLASS_NAMES[c][:12]}" for c in range(len(CLASS_NAMES))],
                   fontsize=8)
ax.set_ylabel("Number of samples")
ax.set_title("Non-IID Data Split: Each client has a different disease distribution", fontsize=12)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print("\nNOTICE: Client 1 sees mostly rare types; Client 2 sees mostly common types.")
print("This is the Non-IID challenge in federated learning!")

## Step 4: Save Client Datasets to Google Drive

We save each client's data as a separate `.npz` file. In a real FL system, each file would live on a different machine. Here they all go to Google Drive so the next notebooks can load them.

In [ ]:
# Save Client 1 data
np.savez(os.path.join(DATA_DIR, "client1.npz"),
         images=train_images[client1_idx],
         labels=train_labels[client1_idx])

# Save Client 2 data
np.savez(os.path.join(DATA_DIR, "client2.npz"),
         images=train_images[client2_idx],
         labels=train_labels[client2_idx])

# Save shared test data (used for global evaluation by the server)
np.savez(os.path.join(DATA_DIR, "test_data.npz"),
         images=data["test_images"],
         labels=data["test_labels"].flatten())

print("Saved to Google Drive:")
print(f"  data/client1.npz   — {len(client1_idx)} samples (Health Center A)")
print(f"  data/client2.npz   — {len(client2_idx)} samples (Health Center B)")
print(f"  data/test_data.npz — {len(data['test_labels'])} samples (shared test set)")
print("\n✅ Done! Open notebook 3_federated_training.ipynb next.")

## Summary

| What we did | Result |
|-------------|--------|
| Visualized the full dataset | Highly imbalanced — class 5 dominates |
| Split into 2 Non-IID clients | Client 1: rare types. Client 2: common types |
| Saved client datasets | `data/client1.npz`, `data/client2.npz` |

**Key insight:** The more different the clients' data distributions are, the harder it is for federated learning to work well. This is the central challenge of real-world FL systems.

➡️ **Next step: `3_federated_training.ipynb`** — train across both clients without sharing their data.